# Glass Rose Demo

A rose window: one half-sector of seeds is rotated and mirrored, then lit as stained glass and cropped to a circle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math_vis import seeds_rose, make_cmap, glass, finish

SEED = 9
SIZE = 420
N_FOLD = 10
RINGS = 4
rng = np.random.default_rng(SEED)
palette = make_cmap(['#1a0b2e', '#4a1c40', '#b83b5e', '#f08a5d', '#f9ed69'])

In [ ]:
# 1. Symmetric seed layout and ids
seeds, ids = seeds_rose(N_FOLD, RINGS, SIZE, SIZE, rng)

fig, ax = plt.subplots(figsize=(7, 7))
scatter = ax.scatter(seeds[:,0], seeds[:,1], c=ids, cmap='tab20', s=15)
ax.set_xlim(0,SIZE); ax.set_ylim(0,SIZE); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(f'Rose seeds: {len(seeds)} points, ids color one half-sector')
plt.tight_layout(); plt.show()

In [ ]:
# 2. Rose color recipe
r = np.hypot(seeds[:,0] - SIZE/2, seeds[:,1] - SIZE/2) / (0.46 * SIZE)
th = np.arctan2(seeds[:,1] - SIZE/2, seeds[:,0] - SIZE/2)
jitter = rng.normal(0, 0.05, ids.max()+1)[ids]
t_cell = np.clip(
    0.55 * r
    + 0.22 * np.cos(N_FOLD * th)
    + 0.10 * np.cos(2 * N_FOLD * th)
    + jitter
    + 0.1,
    0, 1,
)

flat = (palette(t_cell)[:,:3] * 255).astype(np.uint8)
# assign to pixels for preview
from scipy.spatial import cKDTree
X, Y = np.meshgrid(np.arange(SIZE)+0.5, np.arange(SIZE)+0.5)
cell = cKDTree(seeds).query(np.column_stack([X.ravel(), Y.ravel()]))[1]
flat_img = flat[cell].reshape(SIZE, SIZE, 3)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(flat_img, origin='lower'); ax.axis('off')
ax.set_title('Flat rose color recipe')
plt.tight_layout(); plt.show()

In [ ]:
# 3. Lighting stages
img, stages = glass(seeds, t_cell, palette, SIZE, SIZE, rng=rng,
                    facet=0.45, glow=0.4, lead_px=4.5, lead_var=0.35,
                    bevel=0.35, texture=0.04, bloom=0.2, stages=True)

# apply circular crop
yy, xx = np.mgrid[0:SIZE, 0:SIZE]
mask = np.clip((0.47 * SIZE - np.hypot(xx - SIZE/2, yy - SIZE/2)) / 3, 0, 1)[..., None]
for key in stages:
    stages[key] = (stages[key] * mask + np.array([14,11,18]) * (1 - mask)).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for (name, img), ax in zip(stages.items(), axes.ravel()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 4. Final render
final = finish(stages['final'], name='glass_rose.png', grain=2.5, vignette=0.5, display=True)